In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup


# Step 1: Open the file containing all links
with open("all_links.txt", "r") as file:
    urls = [line.strip() for line in file.readlines()]  # Read all links into a list

# Step 2: Loop through each URL and process
car_links = []  # To collect car detail links
count = 0  # Counter for extracted links

for url in urls:
    try:
        print(f"Fetching content for: {url}")
        # Fetch page content
        response = requests.get(url)
        response.raise_for_status()  # Raise exception for HTTP errors
        
        # Parse the content
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Save the current page's content to 'car.html' for reference
        with open("car.html", "w", encoding="utf-8") as file:
            file.write(soup.prettify())
        
        # Extract car detail links from the page
        for link in soup.find_all('a', href=True):
            href = link['href']
            if "/item/" in href:  # OLX car detail pages often contain '/item/' in the URL
                full_url = f"https://www.olx.com.pk{href}" if href.startswith("/") else href
                car_links.append(full_url)
                count += 1
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")

# Step 3: Save all extracted car detail links to a file
with open("car_links.txt", "w") as file:
    for car_link in car_links:
        file.write(f"{car_link}\n")

# Print the results
print("Extracted Car Links:")
for car_link in car_links:
    print(car_link)
print(f"Total number of car links extracted: {count}")


Fetching content for: https://www.olx.com.pk/dha-phase-1_g5000013/cars_c84
Fetching content for: https://www.olx.com.pk/dha-phase-5_g5000017/cars_c84
Extracted Car Links:
https://www.olx.com.pk/item/daihatsu-charade-1986-03234018360-iid-1095326490
https://www.olx.com.pk/item/daihatsu-charade-1986-03234018360-iid-1095326490
https://www.olx.com.pk/item/honda-br-v-2021-iid-1090847556
https://www.olx.com.pk/item/honda-br-v-2021-iid-1090847556
https://www.olx.com.pk/item/honda-vezel-z-sensing-2015-iid-1095044764
https://www.olx.com.pk/item/honda-vezel-z-sensing-2015-iid-1095044764
https://www.olx.com.pk/item/toyota-platz-model-2000-reg-2007-genuine-condition-car-iid-1094731955
https://www.olx.com.pk/item/toyota-platz-model-2000-reg-2007-genuine-condition-car-iid-1094731955
https://www.olx.com.pk/item/daihatsu-mira-202022-iid-1095414718
https://www.olx.com.pk/item/daihatsu-mira-202022-iid-1095414718
https://www.olx.com.pk/item/honda-city-idsi-2023-iid-1093728411
https://www.olx.com.pk/item/h

In [2]:
import os
import requests
from bs4 import BeautifulSoup
import csv

# Step 1: Load car links from the file
with open("car_links.txt", "r") as file:
    car_links = [line.strip() for line in file.readlines()]

# Step 2: Scrape details for each car
car_data = []

for car_url in car_links:
    try:
        response = requests.get(car_url)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Initialize a dictionary to hold car details
        car_info = {}
        car_info['URL'] = car_url

        # Extract price
        car_info['Price'] = soup.find('span', {'aria-label': 'Price'}).text.strip() if soup.find('span', {'aria-label': 'Price'}) else None

        # Extract title
        car_info['Title'] = soup.find('h1', {'class': '_6a5b090c'}).text.strip() if soup.find('h1', {'class': '_6a5b090c'}) else None

        # Extract location
        car_info['Location'] = soup.find('span', {'aria-label': 'Location'}).text.strip() if soup.find('span', {'aria-label': 'Location'}) else None

        # Extract additional attributes from the "Details" section
        details_section = soup.find('div', {'aria-label': 'Details'})
        if details_section:
            for row in details_section.find_all('div', class_='_92439ac7'):
                key = row.find('span').text.strip()
                value = row.find_all('span')[-1].text.strip()
                car_info[key] = value

        # Append the car info to the list
        car_data.append(car_info)
        print(f"Scraped: {car_url}")

    except Exception as e:
        print(f"Error scraping {car_url}: {e}")

# Step 3: Normalize keys to ensure all dictionaries have the same fields
all_keys = set()
for car in car_data:
    all_keys.update(car.keys())

for car in car_data:
    for key in all_keys:
        car.setdefault(key, None)  # Add missing keys with default value None

# Step 4: Save the data to a CSV file
output_file = "car_data.csv"

try:
    with open(output_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(all_keys))
        writer.writeheader()
        writer.writerows(car_data)
    print(f"Data saved to {output_file}")

except PermissionError as e:
    print(f"Permission error: {e}. Retrying with a new file name.")
    new_output_file = "car_data_new.csv"
    with open(new_output_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(all_keys))
        writer.writeheader()
        writer.writerows(car_data)
    print(f"Data saved to {new_output_file}")


Scraped: https://www.olx.com.pk/item/daihatsu-charade-1986-03234018360-iid-1095326490
Scraped: https://www.olx.com.pk/item/daihatsu-charade-1986-03234018360-iid-1095326490
Scraped: https://www.olx.com.pk/item/honda-br-v-2021-iid-1090847556
Scraped: https://www.olx.com.pk/item/honda-br-v-2021-iid-1090847556
Scraped: https://www.olx.com.pk/item/honda-vezel-z-sensing-2015-iid-1095044764
Scraped: https://www.olx.com.pk/item/honda-vezel-z-sensing-2015-iid-1095044764
Scraped: https://www.olx.com.pk/item/toyota-platz-model-2000-reg-2007-genuine-condition-car-iid-1094731955
Scraped: https://www.olx.com.pk/item/toyota-platz-model-2000-reg-2007-genuine-condition-car-iid-1094731955
Scraped: https://www.olx.com.pk/item/daihatsu-mira-202022-iid-1095414718
Scraped: https://www.olx.com.pk/item/daihatsu-mira-202022-iid-1095414718
Scraped: https://www.olx.com.pk/item/honda-city-idsi-2023-iid-1093728411
Scraped: https://www.olx.com.pk/item/honda-city-idsi-2023-iid-1093728411
Scraped: https://www.olx.com

In [56]:
# Importing the pandas library
import pandas as pd

# Reading the CSV file
# Replace 'your_file.csv' with the path to your CSV file
csv_file_path = 'car_data.csv'
df = pd.read_csv(csv_file_path)

# Display the first few rows of the data
print(df.head())
print(df['Price'])


  Version  Power (hp)  Consumption (l/100 km) Interior  \
0     NaN         NaN                     NaN      NaN   
1     NaN         NaN                     NaN      NaN   
2     GLI         NaN                     NaN      NaN   
3     GLI         NaN                     NaN      NaN   
4    2021         NaN                     NaN      NaN   

                          Location Registration city Number of doors Assembly  \
0  Bahria Town Phase 8, Rawalpindi            Lahore             NaN    Local   
1  Bahria Town Phase 8, Rawalpindi            Lahore             NaN    Local   
2  Bahria Town Phase 8, Rawalpindi         Islamabad             NaN    Local   
3  Bahria Town Phase 8, Rawalpindi         Islamabad             NaN    Local   
4  Bahria Town Phase 8, Rawalpindi         Islamabad             NaN    Local   

  Source                      Model Body Type Car documents  \
0    NaN  Civic VTi Oriel Prosmatec     Sedan      Original   
1    NaN  Civic VTi Oriel Prosmatec   

In [ ]:
# Remove 'Rs' and commas, convert to float, and replace invalid entries with NaN
df['Price'] = df['Price'].replace({'Rs': '', ',': ''}, regex=True)

# Convert to numeric and replace invalid values (strings) with NaN
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Replace NaN values with 0
df['Price'].fillna(0, inplace=True)

# Convert the 'Price' column to float
df['Price'] = df['Price'].astype(float)

# Print the updated 'Price' column and its datatype
print("\nUpdated 'Price' column:")
print(df['Price'])

print("\nDatatype of 'Price' column after conversion:", df['Price'].dtype)



Updated 'Price' column:
0        3850000.0
1        3850000.0
2        3985000.0
3        3985000.0
4        3980000.0
           ...    
12553    4450000.0
12554    3150000.0
12555    3150000.0
12556    2250000.0
12557    2250000.0
Name: Price, Length: 12558, dtype: float64

Datatype of 'Price' column after conversion: float64


C:\Users\warda\AppData\Local\Temp\ipykernel_1164\350000793.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Price'].fillna(0, inplace=True)


In [59]:
# Print the count of rows before removing duplicates
print("Count of rows before removing duplicates:", len(df))

# Remove rows with duplicate values across all columns
df = df.drop_duplicates()

# Print the count of rows after removing duplicates
print("Count of rows after removing duplicates:", len(df_no_duplicates))

Count of rows before removing duplicates: 12558
Count of rows after removing duplicates: 5125


In [60]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
# List of categorical columns you want to encode
categorical_columns = [
    'Interior','Version', 'Location', 'Registration city', 'Assembly', 
    'Source', 'Model', 'Body Type', 'Car documents', 
    'Title', 'Color', 'Air Conditioning', 'Make'
]

# Initialize LabelEncoder
encoder = LabelEncoder()

# Encode each categorical column
for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

# Display the DataFrame after Label Encoding
print("\nData after Label Encoding:")
print(df)
print(df['Price'])


Data after Label Encoding:
       Version  Power (hp)  Consumption (l/100 km)  Interior  Location  \
0         1787         NaN                     NaN         5       110   
2          695         NaN                     NaN         5       110   
4          329         NaN                     NaN         5       110   
6         1787         NaN                     NaN         5       110   
8         1787         NaN                     NaN         5       110   
...        ...         ...                     ...       ...       ...   
12548      259         NaN                     NaN         5       117   
12550     1787         NaN                     NaN         2       117   
12552      676         NaN                     NaN         5       117   
12554     1189         NaN                     NaN         5       117   
12556      697         NaN                     NaN         5       117   

       Registration city Number of doors  Assembly  Source  Model  Body Type  \
0  

In [61]:
print(df['Price'])
print("Count of rows before removing duplicates:", len(df))

# Remove rows with duplicate values across all columns
df = df.drop_duplicates()

# Print the count of rows after removing duplicates
print("Count of rows after removing duplicates:", len(df_no_duplicates))

0        3850000.0
2        3985000.0
4        3980000.0
6        9300000.0
8        3150000.0
           ...    
12548     510000.0
12550    1575000.0
12552    4450000.0
12554    3150000.0
12556    2250000.0
Name: Price, Length: 5125, dtype: float64
Count of rows before removing duplicates: 5125
Count of rows after removing duplicates: 5125


In [62]:
print(df['Price'])
# Replace all NaN or null entries with 0
df_filled = df.fillna(0)

# Print the updated DataFrame
print("DataFrame with NaN values replaced by 0:")
print(df_filled)

0        3850000.0
2        3985000.0
4        3980000.0
6        9300000.0
8        3150000.0
           ...    
12548     510000.0
12550    1575000.0
12552    4450000.0
12554    3150000.0
12556    2250000.0
Name: Price, Length: 5125, dtype: float64
DataFrame with NaN values replaced by 0:
       Version  Power (hp)  Consumption (l/100 km)  Interior  Location  \
0         1787         0.0                     0.0         5       110   
2          695         0.0                     0.0         5       110   
4          329         0.0                     0.0         5       110   
6         1787         0.0                     0.0         5       110   
8         1787         0.0                     0.0         5       110   
...        ...         ...                     ...       ...       ...   
12548      259         0.0                     0.0         5       117   
12550     1787         0.0                     0.0         2       117   
12552      676         0.0                

In [64]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
# List of categorical columns you want to encode
categorical_columns = [
    'Interior','Version', 'Location', 'Registration city', 'Assembly', 
    'Source', 'Model', 'Body Type', 'Car documents', 
    'Title', 'Color', 'Air Conditioning', 'Make'
]

# Initialize LabelEncoder
encoder = LabelEncoder()

# Encode each categorical column
for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

# Display the DataFrame after Label Encoding
print("\nData after Label Encoding:")
print(df)
print(df['Price'])


Data after Label Encoding:
       Version  Power (hp)  Consumption (l/100 km)  Interior  Location  \
0         1787         NaN                     NaN         5       110   
2          695         NaN                     NaN         5       110   
4          329         NaN                     NaN         5       110   
6         1787         NaN                     NaN         5       110   
8         1787         NaN                     NaN         5       110   
...        ...         ...                     ...       ...       ...   
12548      259         NaN                     NaN         5       117   
12550     1787         NaN                     NaN         2       117   
12552      676         NaN                     NaN         5       117   
12554     1189         NaN                     NaN         5       117   
12556      697         NaN                     NaN         5       117   

       Registration city Number of doors  Assembly  Source  Model  Body Type  \
0  

In [65]:
# Replace all NaN values in the DataFrame with 0
df = df.fillna(0)

# Display the DataFrame after replacing NaN with 0
print(df)
print(df['Price'])

       Version  Power (hp)  Consumption (l/100 km)  Interior  Location  \
0         1787         0.0                     0.0         5       110   
2          695         0.0                     0.0         5       110   
4          329         0.0                     0.0         5       110   
6         1787         0.0                     0.0         5       110   
8         1787         0.0                     0.0         5       110   
...        ...         ...                     ...       ...       ...   
12548      259         0.0                     0.0         5       117   
12550     1787         0.0                     0.0         2       117   
12552      676         0.0                     0.0         5       117   
12554     1189         0.0                     0.0         5       117   
12556      697         0.0                     0.0         5       117   

       Registration city Number of doors  Assembly  Source  Model  Body Type  \
0                     22       

In [66]:
# Get unique values and the count of unique values for specific columns
columns_to_check = ['Power (hp)', 'Consumption (l/100 km)', 'Number of seats']

for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].unique())  # Print the unique values in the column
    print(f"Count of unique values in '{col}':")
    print(df[col].nunique())  # Print the count of unique values
    print("-" * 50)


Unique values in 'Power (hp)':
[  0.  45.  39. 500.  68. 100. 120. 106.  18. 280.  33.  15. 270.  55.
 176.  13.  10.  66.  11.  37. 240.  20.  80.  54. 200.  28.  52.  60.
  47.  40. 180. 130.]
Count of unique values in 'Power (hp)':
32
--------------------------------------------------
Unique values in 'Consumption (l/100 km)':
[ 0. 20. 17. 16.  7. 14. 13. 12. 15. 18. 10.  6.  4. 19. 11.]
Count of unique values in 'Consumption (l/100 km)':
15
--------------------------------------------------
Unique values in 'Number of seats':
[0. 4. 5. 7. 2. 3. 6.]
Count of unique values in 'Number of seats':
7
--------------------------------------------------


In [67]:
from sklearn.model_selection import train_test_split
# Split the data into 90% training and 10% testing
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# Print the shapes of the resulting DataFrames to verify the split
print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape: {test_df.shape}")
print(df['Price'])

Training set shape: (4612, 20)
Testing set shape: (513, 20)
0        3850000.0
2        3985000.0
4        3980000.0
6        9300000.0
8        3150000.0
           ...    
12548     510000.0
12550    1575000.0
12552    4450000.0
12554    3150000.0
12556    2250000.0
Name: Price, Length: 5125, dtype: float64


In [68]:
print(df['Price'])
# Clean the 'Price' column
df['Price'] = df['Price'].replace({'Rs': ''}, regex=True)  # Remove 'Rs' prefix

# Convert 'Price' column to numeric, forcing errors (invalid values) to NaN
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Replace NaN or invalid values with 0
df['Price'] = df['Price'].fillna(0)

# Ensure all values in 'Price' are floats
df['Price'] = df['Price'].astype(float)

# Display the cleaned 'Price' column
print(df['Price'])

0        3850000.0
2        3985000.0
4        3980000.0
6        9300000.0
8        3150000.0
           ...    
12548     510000.0
12550    1575000.0
12552    4450000.0
12554    3150000.0
12556    2250000.0
Name: Price, Length: 5125, dtype: float64
0        3850000.0
2        3985000.0
4        3980000.0
6        9300000.0
8        3150000.0
           ...    
12548     510000.0
12550    1575000.0
12552    4450000.0
12554    3150000.0
12556    2250000.0
Name: Price, Length: 5125, dtype: float64


In [69]:
df = df.apply(pd.to_numeric, errors='coerce')

# Replace NaN values with 0
df = df.fillna(0)

# Ensure all values are float type
df = df.astype(float)
# Replace '4/5' string in the 'Assembly' column with 0
df['Assembly'] = df['Assembly'].replace('4/5', 0)



In [ ]:

import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error


# Replace '4/5' string in the 'Assembly' column with 0
df['Assembly'] = df['Assembly'].replace('4/5', 0)

# Convert all columns to numeric, errors='coerce' turns invalid strings to NaN
df = df.apply(pd.to_numeric, errors='coerce')

# Replace NaN values with 0 (or any other appropriate value)
df = df.fillna(0)

# Ensure all values are float type
df = df.astype(float)

# Split the data into features (X) and target (y)
y = df['Price']
X = df.drop(columns=['Price'])  # Assuming 'Price' is the target column



Target (y):
0        3850000.0
2        3985000.0
4        3980000.0
6        9300000.0
8        3150000.0
           ...    
12548     510000.0
12550    1575000.0
12552    4450000.0
12554    3150000.0
12556    2250000.0
Name: Price, Length: 5125, dtype: float64


In [ ]:
# Initialize the Decision Tree model
tree_model = DecisionTreeRegressor(random_state=42)

# Perform 10-fold cross-validation
cv_scores = cross_val_score(tree_model, X, y, cv=10, scoring='neg_mean_squared_error')

# Calculate and display the average negative MSE and its positive equivalent
average_mse = cv_scores.mean()
print(f"Average MSE from 10-fold cross-validation: {average_mse}")

# You can also print the MSE for each fold
print(f"MSE for each fold: {cv_scores}")

Average MSE from 10-fold cross-validation: 10206617151631.607
MSE for each fold: [-1.66729635e+13 -5.66779889e+12 -5.53685128e+12 -2.04177236e+13
 -7.24686039e+12 -6.19538721e+12 -1.47930661e+13 -7.71026513e+12
 -7.13361117e+12 -1.06916443e+13]


In [76]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, f1_score, mean_squared_error
import joblib

# Assuming df is your DataFrame and 'Price' is the target column
y = df['Price']
X = df.drop(columns=['Price'])

# Split the data into training and testing sets (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Initialize the best model and the best score
best_model = None
best_score = float('inf')

# Dictionary to store metrics for each epoch
metrics = []

# Train the model for 10 epochs (iterations)
for epoch in range(10):
    # Initialize the Decision Tree model (you can tune hyperparameters if needed)
    tree_model = DecisionTreeRegressor(random_state=42)
    
    # Fit the model on the training data
    tree_model.fit(X_train, y_train)
    
    # Make predictions on the test set
    y_pred = tree_model.predict(X_test)
    
    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=1)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=1)
    
    # Print the metrics for each epoch
    print(f"Epoch {epoch+1}: MSE = {mse}, Accuracy = {accuracy}, Balanced Accuracy = {balanced_accuracy}, Recall = {recall}, F1 Score = {f1}")
    
    # Save the model if it's the best one so far
    if mse < best_score:
        best_score = mse
        best_model = tree_model
        print(f"New best model found with MSE = {best_score}")
    
    # Save the metrics for this epoch
    metrics.append({
        'Epoch': epoch + 1,
        'MSE': mse,
        'Accuracy': accuracy,
        'Balanced Accuracy': balanced_accuracy,
        'Recall': recall,
        'F1 Score': f1
    })

# Save the best model to a .pkl file
joblib.dump(best_model, 'best_decision_tree_model.pkl')

print("Best model saved as 'best_decision_tree_model.pkl'")

# Save all metrics to a CSV file
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv('model_metrics.csv', index=False)

print("Metrics saved as 'model_metrics.csv'")


C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Epoch 1: MSE = 12199162205527.41, Accuracy = 0.03313840155945419, Balanced Accuracy = 0.03051801801801802, Recall = 0.03313840155945419, F1 Score = 0.03336641699214798
New best model found with MSE = 12199162205527.41
Epoch 2: MSE = 12199162205527.41, Accuracy = 0.03313840155945419, Balanced Accuracy = 0.03051801801801802, Recall = 0.03313840155945419, F1 Score = 0.03336641699214798


C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Epoch 3: MSE = 12199162205527.41, Accuracy = 0.03313840155945419, Balanced Accuracy = 0.03051801801801802, Recall = 0.03313840155945419, F1 Score = 0.03336641699214798
Epoch 4: MSE = 12199162205527.41, Accuracy = 0.03313840155945419, Balanced Accuracy = 0.03051801801801802, Recall = 0.03313840155945419, F1 Score = 0.03336641699214798


C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


...

C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Epoch 7: MSE = 12199162205527.41, Accuracy = 0.03313840155945419, Balanced Accuracy = 0.03051801801801802, Recall = 0.03313840155945419, F1 Score = 0.03336641699214798
Epoch 8: MSE = 12199162205527.41, Accuracy = 0.03313840155945419, Balanced Accuracy = 0.03051801801801802, Recall = 0.03313840155945419, F1 Score = 0.03336641699214798


C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Epoch 9: MSE = 12199162205527.41, Accuracy = 0.03313840155945419, Balanced Accuracy = 0.03051801801801802, Recall = 0.03313840155945419, F1 Score = 0.03336641699214798
Epoch 10: MSE = 12199162205527.41, Accuracy = 0.03313840155945419, Balanced Accuracy = 0.03051801801801802, Recall = 0.03313840155945419, F1 Score = 0.03336641699214798
Best model saved as 'best_decision_tree_model.pkl'
Metrics saved as 'model_metrics.csv'


C:\Users\warda\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
